# Training Changer-r18 with Open-CD

[Changer](https://github.com/likyoo/open-cd#changer) is a family of change detection models that decouple the encoding and interaction of bitemporal image features. Changer-r18 uses a ResNet-18 backbone and achieves a strong accuracy/speed tradeoff on LEVIR-CD.

Open-CD uses mmengine's config system for training. This notebook:
1. Builds a Changer-r18 model programmatically
2. Trains it on the LEVIR-CD mini dataset using mmengine's `Runner`
3. Logs metrics to TensorBoard

## References
- Changer paper: https://arxiv.org/abs/2209.08290
- Open-CD configs: https://github.com/likyoo/open-cd/tree/main/configs/changer

## 1. Define the Config

In [ ]:
import os
from mmengine.config import Config

dataset_dir = "LEVIR-CD-mini"
assert os.path.exists(dataset_dir), "Run 1-levir_cd_dataset.ipynb first."

# Build config programmatically (equivalent to a .py config file)
cfg_dict = dict(
    # Model
    model=dict(
        type="opencd.Changer",
        data_preprocessor=dict(
            type="DualInputSegDataPreProcessor",
            mean=[123.675, 116.28, 103.53] * 2,
            std=[58.395, 57.12, 57.375] * 2,
            bgr_to_rgb=True,
            pad_val=0,
            seg_pad_val=255,
            size=(256, 256),
        ),
        backbone=dict(
            type="opencd.interaction_resnet.InteractionResNet",
            encoder=dict(
                type="ResNet",
                depth=18,
                out_indices=(0, 1, 2, 3),
                pretrained="open-mmlab://resnet18_v1c",
            ),
        ),
        decode_head=dict(
            type="opencd.TinyHead",
            in_channels=[64, 128, 256, 512],
            num_classes=2,
        ),
        train_cfg=dict(),
        test_cfg=dict(mode="whole"),
    ),
    # Dataset
    dataset_type="opencd.LEVIRCDDataset",
    data_root=dataset_dir,
    train_dataloader=dict(
        batch_size=4,
        num_workers=2,
        dataset=dict(
            type="opencd.LEVIRCDDataset",
            data_root=dataset_dir,
            data_prefix=dict(
                img_path="train/A",
                img_path2="train/B",
                seg_map_path="train/label",
            ),
            pipeline=[
                dict(type="MultiImgLoadImageFromFile"),
                dict(type="MultiImgLoadAnnotations", reduce_zero_label=False),
                dict(type="MultiImgRandomFlip", prob=0.5),
                dict(type="MultiImgPackSegInputs"),
            ],
        ),
    ),
    val_dataloader=dict(
        batch_size=4,
        num_workers=2,
        dataset=dict(
            type="opencd.LEVIRCDDataset",
            data_root=dataset_dir,
            data_prefix=dict(
                img_path="val/A",
                img_path2="val/B",
                seg_map_path="val/label",
            ),
            pipeline=[
                dict(type="MultiImgLoadImageFromFile"),
                dict(type="MultiImgLoadAnnotations", reduce_zero_label=False),
                dict(type="MultiImgPackSegInputs"),
            ],
        ),
    ),
    # Scheduler
    optim_wrapper=dict(
        optimizer=dict(type="AdamW", lr=6e-5, betas=(0.9, 0.999), weight_decay=0.01)
    ),
    param_scheduler=[
        dict(type="LinearLR", start_factor=1e-6, by_epoch=False, begin=0, end=500),
        dict(type="PolyLR", eta_min=0.0, power=1.0, begin=500, end=4000, by_epoch=False),
    ],
    train_cfg=dict(type="IterBasedTrainLoop", max_iters=4000, val_interval=500),
    val_cfg=dict(type="ValLoop"),
    val_evaluator=dict(type="IoUMetric", iou_metrics=["mIoU", "mFscore"]),
    # Runner
    work_dir="changer_output",
    default_hooks=dict(
        timer=dict(type="IterTimerHook"),
        logger=dict(type="LoggerHook", interval=50),
        checkpoint=dict(type="CheckpointHook", by_epoch=False, interval=1000),
    ),
    visualizer=dict(type="SegLocalVisualizer", vis_backends=[dict(type="TensorboardVisBackend")]),
    launcher="none",
)

cfg = Config(cfg_dict)
print("Config built successfully.")
print(f"Work directory: {cfg.work_dir}")

## 2. Alternative: Use Official Config Files

For production use, load the official config file directly:

In [ ]:
import opencd

print("Using official Open-CD config files:")
print("""
# 1. Clone Open-CD source
git clone https://github.com/likyoo/open-cd.git

# 2. Train using the CLI runner
python open-cd/tools/train.py open-cd/configs/changer/changer_r18_512x512_40k_levircd.py \\
    --work-dir ./changer_work_dir

# 3. Or load from notebook:
from mmengine.config import Config
cfg = Config.fromfile('open-cd/configs/changer/changer_r18_512x512_40k_levircd.py')
# Override data root:
cfg.data_root = '/path/to/LEVIR-CD-256'
""")

## 3. Build and Inspect the Model

In [ ]:
from mmengine.registry import MODELS
import torch

model = MODELS.build(cfg.model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Changer-r18 parameters:")
print(f"  Total:     {total_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"  Size (FP32): {total_params * 4 / 1e6:.1f} MB")

## 4. Train with mmengine Runner

> **Note:** Full training on LEVIR-CD takes 4000 iterations (~2–4 hours on GPU). The cell below runs a short sanity-check (10 iterations) to verify the setup. For real training, increase `max_iters` and use GPU.

In [ ]:
from mmengine.runner import Runner

# Override for quick sanity check
cfg.train_cfg.max_iters = 10
cfg.train_cfg.val_interval = 10
cfg.param_scheduler[0].end = 5
cfg.param_scheduler[1].begin = 5
cfg.param_scheduler[1].end = 10

runner = Runner.from_cfg(cfg)
print("Runner initialized. Starting training...")
runner.train()
print("\nSanity check complete. Increase max_iters for real training.")

## 5. Launch TensorBoard

In [ ]:
print("To view training metrics in TensorBoard, run:")
print("  tensorboard --logdir changer_output/vis_data/ --port 1235")
print("\nThen open http://localhost:1235 in your browser.")
print("(Port 1235 is pre-forwarded in the devcontainer.)")

## 6. Model Zoo — Published Results on LEVIR-CD

| Model | F1 | IoU | Params | FPS |
|---|---|---|---|---|
| FC-EF | 83.4 | 71.5 | 1.4M | 74.5 |
| SNUNet-c16 | 88.2 | 78.8 | 12.0M | 36.1 |
| ChangeFormer | 90.4 | 82.4 | 41.0M | 15.9 |
| Changer-r18 | 91.4 | 84.2 | 14.1M | 44.8 |
| Changer-s101 | 92.5 | 86.1 | 50.1M | 8.3 |
| BAN-vit-l16 | 93.5 | 87.8 | 336M | 3.2 |

Source: https://github.com/likyoo/open-cd (results on 256x256 patches)

Continue to `3-inference_and_visualization.ipynb` to run inference on a trained checkpoint.